# The Evolution of Post-Training

> Pretraining gives a model the ability to continue text, but not necessarily to answer questions well, follow instructions, or reason in the requested form. All training that turns a raw pretrained model into a useful assistant is called post-training. It changes the model's output distribution under different contexts rather than its architecture.
>
> Two mature paradigms dominate post-training. SFT imitates an external dataset and provides dense signals, but trains off the model's own distribution. RL trains on the model's behavior, but often reduces feedback to one scalar reward.
>
> **OPD** (On-Policy Distillation) combines part of each advantage: the student generates its own trajectory, and a teacher corrects the student's next-token distribution at every prefix on that trajectory.
>
> We begin with this evolution of paradigms, then study OPD's KL direction, OPSD variants, signal granularity, KL estimators, and a complete training loop.


## 1. The Evolution of Post-Training Paradigms

To compare SFT, RL, and OPD, start from one basic view: a language model generates a probability distribution over the vocabulary for each context. Post-training reshapes that distribution, increasing the probability of desired behavior and reducing the probability of undesired behavior.


### 1.1 Model Output Distributions

To understand the connections and differences among SFT, RL, and OPD, start from the most basic concept: **a language model is essentially a probability distribution generator**. Given a context, the model assigns a probability to every token in the vocabulary -- this is the model's output distribution at that context.

What post-training does is reshape this probability distribution: raising the probability of some behaviors and lowering others.

Build this intuition first, then see how each of the three methods does it.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'DejaVu Sans'

def softmax(logits):
    logits = np.array(logits, dtype=np.float64)
    e = np.exp(logits - logits.max())
    return e / e.sum()

# A small vocabulary
vocab = ["the", "to", "is", "fix", "func", "code", "Python", "math", "bug", "output"]
vocab_size = len(vocab)

# Logits from the same model under two different contexts
logits_chat = np.array([0.8, 0.5, 0.6, -2.0, -1.5, -1.8, -1.5, -1.2, -2.5, -1.0])
logits_code = np.array([-1.5, -1.8, -1.2, 0.8, 0.6, 1.2, 0.9, -1.0, -0.5, 0.3])

probs_chat = softmax(logits_chat)
probs_code = softmax(logits_code)

# Visualize how the same model under different contexts produces different distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

colors_chat = ['#4472C4' if p > 0.05 else '#D6E4F0' for p in probs_chat]
ax1.bar(vocab, probs_chat, color=colors_chat, edgecolor='white')
ax1.set_title("Context: weather and going out", fontsize=13)
ax1.set_ylabel("Probability", fontsize=12)
ax1.tick_params(axis='x', rotation=45)

colors_code = ['#ED7D31' if p > 0.05 else '#FCE4D6' for p in probs_code]
ax2.bar(vocab, probs_code, color=colors_code, edgecolor='white')
ax2.set_title("Context: code repair prompt", fontsize=13)
ax2.set_ylabel("Probability", fontsize=12)
ax2.tick_params(axis='x', rotation=45)

plt.suptitle("A language model is a probability distribution generator", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print(f"-> The same model under different contexts produces noticeably different probability distributions")
print(f"-> Post-training reshapes the distribution within a particular context toward the behavior we want")


### 1.2 SFT and External Data Distributions

SFT (Supervised Fine-Tuning) is the most intuitive post-training method. First, prepare a labeled dataset. During training, the model sees the prompt and the gold answer, and learns through Cross-Entropy: at each position where a gold answer token appears, the model increases its probability of generating that token.

From a distribution perspective, SFT's target distribution comes from an external dataset. Its strength is directness -- especially suitable for cold starting. When the model cannot yet follow instructions, SFT quickly builds basic behavioral templates.

But SFT's risk also comes from here: every token in the gold answer is treated as a learning target -- critical reasoning tokens must be learned, stylistic words in the dataset must also be learned, and even incidental expressions and template quirks get picked up. SFT loss itself does not distinguish these differences; it pushes the entire answer upward uniformly.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === SFT: simulate CE gradient descent, pulling the model distribution toward the external target distribution ===
print("=== SFT: Imitating External Data Distributions ===\n")

# Pretrained model's output distribution at a code-fixing context (biased toward common general words)
init_logits = np.array([1.8, 1.5, 1.6, -1.0, -0.8, -0.5, -0.5, -0.3, -0.8, -0.5])
init_model = softmax(init_logits)

# Target distribution from external labeled data (task-relevant tokens have higher probability)
target = np.array([0.02, 0.01, 0.02, 0.25, 0.20, 0.15, 0.20, 0.10, 0.03, 0.02])

# Simulate CE gradient descent: dCE/dlogits = softmax(logits) - target
model_logits = init_logits.copy()
lr = 0.3
steps = 12
history = [softmax(model_logits)]

for step in range(steps):
    model_probs = softmax(model_logits)
    grad = model_probs - target          # CE gradient w.r.t. logits
    model_logits = model_logits - lr * grad  # Gradient descent
    history.append(softmax(model_logits))

final_model = history[-1]

# Visualization: before SFT vs target distribution vs after SFT
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

axes[0].bar(vocab, init_model, color='#4472C4', edgecolor='white')
axes[0].set_title('Before SFT (pretrained model)', fontsize=13)
axes[0].set_ylabel('Probability', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(init_model):
    if v > 0.05:
        axes[0].text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)

axes[1].bar(vocab, target, color='#ED7D31', edgecolor='white')
axes[1].set_title('Target distribution from external dataset', fontsize=13)
axes[1].tick_params(axis='x', rotation=45)
for i, v in enumerate(target):
    if v > 0.05:
        axes[1].text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)

axes[2].bar(vocab, final_model, color='#70AD47', edgecolor='white')
axes[2].set_title(f'After SFT ({steps} gradient steps)', fontsize=13)
axes[2].tick_params(axis='x', rotation=45)
for i, v in enumerate(final_model):
    if v > 0.05:
        axes[2].text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)

plt.suptitle('SFT: cross-entropy pulls the model\ntoward the target', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# Print key changes
ce_init = -np.sum(target * np.log(init_model + 1e-10))
ce_final = -np.sum(target * np.log(final_model + 1e-10))
print(f"CE Loss: {ce_init:.4f} → {ce_final:.4f}")
print(f"\nKey observations:")
print(f"  'fix' probability: {init_model[3]:.3f} -> {final_model[3]:.3f} (target={target[3]:.3f}) ↑")
print(f"  'code' probability: {init_model[5]:.3f} -> {final_model[5]:.3f} (target={target[5]:.3f}) ↑")
print(f"  'the' probability:  {init_model[0]:.3f} -> {final_model[0]:.3f} (target={target[0]:.3f}) ↓")
print(f"\n-> SFT pulls the entire distribution toward external data -- task tokens rise, common general words are suppressed")
print(f"-> Risk: all tokens in external data (including stylistic words, incidental expressions) are treated equally")
print(f"-> If the external distribution is far from the original model, old capabilities may be overwritten (catastrophic forgetting)")


### 1.3 RL and Reward-Based Selection

RL's training logic is fundamentally different from SFT. The model first generates responses from its current policy, then a reward function scores these responses -- high-reward responses become more likely to be generated in the future, while low-reward ones are suppressed.

The key difference is that RL's samples come from the model's own current generation distribution, not from an external dataset. This means RL **does not pull the model toward an arbitrarily distant external data distribution, but instead pushes probability mass toward higher-reward behaviors within the regions the model actually visits**.

This is also why RL is less prone to forgetting. But it has an obvious problem: rewards are often sparse -- when an answer is wrong, we only know the entire trajectory is bad, but not necessarily which step caused the error.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === RL: simulate REINFORCE policy gradient, selecting high-reward behaviors from own samples ===
print("=== RL: Selecting high-reward directions from model-generated behaviors ===\n")

np.random.seed(42)

# Initial model distribution (at a code-fixing context)
init_logits = np.array([-1.0, -1.2, -0.8, 0.8, 0.6, 1.2, 0.5, -0.5, -0.3, 0.1])
init_probs = softmax(init_logits)

# True task value for each token: whether the code is correct, tests pass
true_reward = np.array([0, 0, 0, 8, 6, 7, 5, 3, 4, 2])  # matching vocab order

# Simulate REINFORCE: d(-log pi(a) * R) / dlogits = -R * (one_hot(a) - pi)
current_logits = init_logits.copy()
lr = 0.08
n_rounds = 8
n_samples = 20

history = [softmax(current_logits)]

for round_idx in range(n_rounds):
    probs = softmax(current_logits)
    gradient = np.zeros(vocab_size)
    
    for _ in range(n_samples):
        a = np.random.choice(vocab_size, p=probs)
        r = true_reward[a]
        # REINFORCE gradient (negative reward direction, since we minimize negative reward)
        one_hot = np.zeros(vocab_size)
        one_hot[a] = 1
        gradient += -r * (one_hot - probs)
    
    gradient /= n_samples
    current_logits = current_logits - lr * gradient
    history.append(softmax(current_logits))

final_probs = history[-1]

# Visualization: before RL vs after RL, colored by reward
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

reward_norm = (true_reward - true_reward.min()) / (true_reward.max() - true_reward.min() + 1e-10)
colors = [plt.cm.RdYlGn(0.2 + 0.6 * rn) for rn in reward_norm]

ax1.bar(vocab, init_probs, color=colors, edgecolor='white')
ax1.set_title('Before RL (initial policy distribution)', fontsize=13)
ax1.set_ylabel('Probability', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
for i, v in enumerate(init_probs):
    if v > 0.05:
        ax1.text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)

ax2.bar(vocab, final_probs, color=colors, edgecolor='white')
ax2.set_title(f'After RL ({n_rounds} rounds x {n_samples} samples/round)', fontsize=13)
ax2.set_ylabel('Probability', fontsize=12)
ax2.tick_params(axis='x', rotation=45)
for i, v in enumerate(final_probs):
    if v > 0.05:
        ax2.text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)

plt.suptitle('RL: policy gradient increases\nhigh-reward token probability', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("Probability changes (sorted by reward, green=high value, red=low value):")
for i in np.argsort(true_reward)[::-1]:
    change = final_probs[i] - init_probs[i]
    direction = "↑" if change > 0.005 else ("↓" if change < -0.005 else "→")
    print(f"  {vocab[i]:6s} (reward={true_reward[i]}): "
          f"{init_probs[i]:.3f} → {final_probs[i]:.3f} ({change:+.3f}) {direction}")

print(f"\n-> on-policy: samples come from the model's own current generation distribution")
print(f"-> Updates only happen on tokens the model actually sampled -> won't pull it to distant external distributions")
print(f"-> High-reward tokens gradually increase in probability, low-reward tokens are suppressed")
print(f"-> But reward is outcome-level (sparse) -- the model doesn't know 'which step' caused good/bad results")


### 1.4 A Fourth Path: OPD

OPD can be seen as a combination between SFT and RL. Like SFT/distillation, it has a teacher signal; like RL, the training data comes from the student's own current generation distribution.

The core flow is: the student generates its own response -> the teacher provides the next-token distribution at each prefix the student generated -> the student updates to make its distribution closer to the teacher's.

This is fundamentally different from standard offline distillation: in offline distillation, the teacher walks a standard route and the student follows along; in OPD, the student walks first, and wherever it goes, the teacher corrects it on the spot.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === OPD: student does its own rollout, teacher gives token-level corrections at student's prefixes ===
print("=== OPD: Student walks on its own, Teacher gives dense guidance where the student arrives ===\n")

np.random.seed(42)

# Student model -- initially weaker, more spread-out distribution
student_base_logits = np.array([0.7, 0.2, 0.3, -0.5, -0.3, -0.1, -0.2, -0.4, -0.6, -0.4])
# Teacher model -- stronger, more confident distribution (lower temperature)
teacher_base_logits = np.array([-0.5, -0.8, -0.4, 1.2, 0.8, 1.5, 0.9, -0.5, -0.3, 0.3])

s_probs = softmax(student_base_logits)
t_probs = softmax(teacher_base_logits / 0.6)  # teacher lower temperature -> sharper distribution

# 1. Student autoregressively generates 3 steps (simulating slight distribution shifts at different prefixes)
print("1. Student autoregressively generates 3 steps:")
seq = []
positions_logits = []
for step in range(3):
    # Each step's logits shift slightly from the base (simulating prefix changes)
    step_logits = student_base_logits + np.random.RandomState(step * 7).randn(vocab_size) * 0.4
    positions_logits.append(step_logits)
    step_probs = softmax(step_logits)
    tok_id = np.random.choice(vocab_size, p=step_probs)
    seq.append(tok_id)
    print(f"   Step {step+1}: chose '{vocab[tok_id]}' (most confident about '{vocab[np.argmax(step_probs)]}')")

# 2. Teacher provides per-token distribution at each student prefix
print(f"\n2. Teacher provides full token distribution at each student prefix:")
total_kl = 0
for t in range(3):
    s_p = softmax(positions_logits[t])
    t_p = softmax(teacher_base_logits / 0.6)
    kl = np.sum(s_p * (np.log(s_p + 1e-10) - np.log(t_p + 1e-10)))
    total_kl += kl
    print(f"   Position {t}: student chose '{vocab[seq[t]]}', "
          f"teacher most confident about '{vocab[np.argmax(t_p)]}', "
          f"KL(s||t) = {kl:.4f}")

# 3. Simulate OPD gradient update: minimize KL(student || teacher)
# dKL(s||t)/ds_logits = s_probs - t_probs (when t is fixed)
# This is the key to OPD: pull student distribution toward teacher, but only at positions where student probability is high
s_logits = student_base_logits.copy()
lr = 0.15
opd_steps = 10
opd_history = [softmax(s_logits)]

for step in range(opd_steps):
    s_p = softmax(s_logits)
    grad = s_p - t_probs           # KL(s||t) gradient w.r.t. s_logits
    s_logits = s_logits - lr * grad  # Gradient descent
    opd_history.append(softmax(s_logits))

s_final = opd_history[-1]

# Visualization: Student before vs Teacher vs Student after
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

axes[0].bar(vocab, softmax(student_base_logits), color='#4472C4', edgecolor='white')
axes[0].set_title('Before OPD (initial student distribution)', fontsize=13)
axes[0].set_ylabel('Probability', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(vocab, t_probs, color='#ED7D31', edgecolor='white')
axes[1].set_title('Teacher distribution (target)', fontsize=13)
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(vocab, s_final, color='#70AD47', edgecolor='white')
axes[2].set_title(f'After OPD ({opd_steps} KL minimization steps)', fontsize=13)
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle('OPD: KL minimization moves the student toward the teacher'
             'on its own high-probability region', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n  Total KL loss = {total_kl:.4f}")
print(f"\n-> OPD characteristics:")
print(f"  on-policy: prefixes come from student's own generation -> training distribution is in regions the model actually visits")
print(f"  dense signal: teacher distribution guidance at every token position -> unlike RL which only gives outcome reward")
print(f"  mode-seeking: KL(s||t) only aligns where student probability is high -> preserves patterns the student is good at")
print(f"  More resistant to forgetting than SFT (data from own distribution), denser signal than RL (per-token guidance)")


### 1.5 Comparing the Paradigms

Looking at SFT, RL, and OPD together, the core difference is not about "is there a teacher" or "is there a reward", but about data source and signal density. The table below summarizes the key differences from a distribution perspective.


Just two questions distinguish all methods:

1. **Who writes the prefix during training?**
2. **Who provides the learning target?**

| Method | Prefix source | Target source | Analogy |
|:---|:---|:---|:---|
| **SFT** | Dataset gold answers | Dataset gold answers | Memorizing the textbook |
| **Offline distillation** | Teacher-written fixed prefixes | Teacher's probability distribution | Memorizing teacher's model essays |
| **OPD** | **Student-generated prefixes** | Teacher's distribution at these student prefixes | Student writes, teacher grades |
| **OPSD** | **Student-generated prefixes** | Same model's open-book version | Closed-book student, taught by open-book self |

The core difference is in the prefix column. In OPD and OPSD, the prefix is written by the student; in all others, it is written by someone else.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === Core differences: visual comparison ===
print("=== One table: SFT vs RL vs OPD (distribution perspective) ===\n")

header = f"{'':22} {'SFT':^18} {'RL':^18} {'OPD':^18}"
print(header)
print("-" * 76)

rows = [
    ("Training data source", "External dataset", "Student-generated", "Student-generated"),
    ("On-policy?", "No", "Yes", "Yes"),
    ("Signal density", "Per-token (dense)", "Outcome (sparse)", "Per-token (dense)"),
    ("Target source", "Labeled answers", "Reward-defined value", "Teacher token dist"),
    ("Distribution shift", "Pull to external", "Filter in current", "Local correction"),
    ("Strength", "Cold start, formatting", "Verifiable tasks", "On-policy + dense"),
    ("Risk", "Over-pulling, forgetting", "Sparse reward, cost", "Teacher bias"),
]

for r in rows:
    print(f"{r[0]:22} {r[1]:^18} {r[2]:^18} {r[3]:^18}")

print(f"\n-> The three are not replacements, but three different 'distribution shifting methods'")

# Visualization: schematic 'distribution heatmap' comparing the three methods
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Build a 2D schematic distribution grid (simulating token space)
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)

# Original model distribution (Gaussian centered upper-left)
mu_orig = np.array([-0.8, 0.5])
sigma = np.array([[1.2, -0.3], [-0.3, 1.5]])
Z_orig = np.exp(-0.5 * ((X - mu_orig[0])**2 / sigma[0,0] 
                         + (Y - mu_orig[1])**2 / sigma[1,1]
                         - 2 * (X - mu_orig[0]) * (Y - mu_orig[1])
                         * sigma[0,1] / (sigma[0,0]*sigma[1,1])))

# -- SFT: pull overall toward external target (upper-right) --
mu_target = np.array([1.2, 0.3])
Z_target = np.exp(-0.5 * ((X - mu_target[0])**2 / 0.8 + (Y - mu_target[1])**2 / 0.8))
Z_sft = 0.3 * Z_orig + 0.7 * Z_target  # After SFT: overall shift toward target

ax = axes[0]
ax.contourf(X, Y, Z_orig, levels=8, cmap='Blues', alpha=0.5)
ax.contourf(X, Y, Z_sft, levels=8, cmap='Oranges', alpha=0.5)
ax.arrow(mu_orig[0], mu_orig[1], mu_target[0]-mu_orig[0], mu_target[1]-mu_orig[1],
         head_width=0.25, head_length=0.2, fc='red', ec='red', linewidth=2)
ax.set_title('SFT: pull toward external distribution', fontsize=13)
ax.set_xticks([]); ax.set_yticks([])

# -- RL: filter for high-reward sub-regions within original distribution --
Z_rl = Z_orig.copy()
# Enhance signal in high-reward region (upper-right)
reward_mask = (X > 0) & (Y > -1) & (Y < 1.5)
Z_rl[reward_mask] *= 2.0
Z_rl[~reward_mask] *= 0.5
Z_rl = Z_rl / Z_rl.sum()

ax = axes[1]
ax.contourf(X, Y, Z_orig, levels=8, cmap='Blues', alpha=0.5)
ax.contourf(X, Y, Z_rl, levels=8, cmap='Greens', alpha=0.5)
ax.set_title('RL: select high-reward directions\ninside its own distribution', fontsize=13)
ax.set_xticks([]); ax.set_yticks([])

# -- OPD: locally correct in high-probability regions of own distribution --
Z_opd = Z_orig.copy()
# Fine-tune toward target in model's own high-probability region (near center)
local_mask = (Z_orig > Z_orig.max() * 0.3)
shift = Z_target * 0.3
Z_opd[local_mask] = Z_orig[local_mask] * 0.7 + shift[local_mask]
Z_opd = Z_opd / Z_opd.sum()

ax = axes[2]
ax.contourf(X, Y, Z_orig, levels=8, cmap='Blues', alpha=0.5)
ax.contourf(X, Y, Z_opd, levels=8, cmap='Purples', alpha=0.5)
ax.set_title('OPD: teacher corrects each token\non student trajectories', fontsize=13)
ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('How three post-training methods move distributions\n'
             '(blue=original distribution, orange/green/purple=updated distribution)',
             fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print(f"\n-> SFT: External answers tell the model 'this is what the answer should look like'")
print(f"-> RL: The environment tells the model 'which of your own behaviors are more valuable'")
print(f"-> OPD: The teacher tells the model 'when you arrive here, this is how you should proceed next'")


In [ ]:
import numpy as np

np.random.seed(42)

# Simulated vocabulary
vocab = {"I": 0, "yesterday": 1, "went": 2, "school": 3, "park": 4, "store": 5, "very": 6, "happy": 7, "bored": 8, "tired": 9}
id2token = {v: k for k, v in vocab.items()}
vocab_size = len(vocab)

def softmax(logits):
    logits = np.array(logits, dtype=np.float64)
    exp_logits = np.exp(logits - logits.max())
    return exp_logits / exp_logits.sum()

def log_softmax(logits):
    logits = np.array(logits, dtype=np.float64)
    m = logits.max()
    return logits - m - np.log(np.exp(logits - m).sum())

# Simulated student model: outputs logits based on input sequence
def student_model(input_ids):
    rng = np.random.RandomState(sum(input_ids) * 7 + 42)
    return rng.randn(vocab_size) * 2

# Simulated teacher model: smarter, more confident
def teacher_model(input_ids):
    rng = np.random.RandomState(sum(input_ids) * 13 + 7)
    return rng.randn(vocab_size) * 1.2

print("Vocabulary and model simulation functions ready!")


The central question is now precise: how can OPD combine trajectories from the student's own policy with dense token-level supervision?

The next section explains why the first half matters. SFT's Exposure Bias is the starting point for understanding every on-policy method.


## 2. Exposure Bias

This is one of the key concepts for understanding OPD's motivation.

```
During training (SFT / offline distillation):
  The prefix is always the first half of the gold answer
  e.g., "I went yesterday" -> learn to predict "school"
  The student only sees grammatically correct, semantically coherent prefixes

During inference (student generates on its own):
  The student may produce nonsensical prefixes
  e.g., "I yesterday very park" -> now predict the next word?
  The student has never seen "I yesterday very park" as a prefix!
  -> May become unstable or continue to deviate
```

**This is Exposure Bias**: during training you stand on the "correct track", but during inference, once you deviate, you enter unknown territory.

It's like only practicing driving on smooth highways, then being more likely to make mistakes the first time you hit a dirt road.


In [ ]:
import numpy as np

# Intuitive demonstration of Exposure Bias: using simulated models to show training vs inference discrepancy
print("=== Exposure Bias Demonstration ===")
print()

# Training prefix: first half of the gold answer
train_prefix = [0, 1, 2]  # "I went yesterday"
print(f"Training prefix: {[id2token[i] for i in train_prefix]} <- from dataset, grammatically correct")
t_logits = teacher_model(train_prefix)
t_probs = softmax(t_logits)
top3 = np.argsort(t_probs)[-3:][::-1]
print("  Student trained on this prefix, teacher's top 3 preferred words:")
for idx in top3:
    print(f"    '{id2token[idx]}': {t_probs[idx]:.3f}")
print()

# Inference: student generates on its own, may deviate
print("During inference, student autoregressively generates:")
np.random.seed(99)
student_sequence = [0]  # "I"
for step in range(3):
    s_logits = student_model(student_sequence)
    s_probs = softmax(s_logits)
    next_token = np.random.choice(vocab_size, p=s_probs)
    student_sequence.append(next_token)
    print(f"  Step {step+1}: Input {[id2token[i] for i in student_sequence[:-1]]} "
          f"-> sampled '{id2token[next_token]}' (probability={s_probs[next_token]:.3f})")
print()

# Check model confusion under the student's nonsensical prefix
weird_prefix = student_sequence[:-1]
s_logits_weird = student_model(weird_prefix)
s_probs_weird = softmax(s_logits_weird)
entropy = -np.sum(s_probs_weird * np.log(s_probs_weird + 1e-10))
print(f"Student-generated prefix: {[id2token[i] for i in weird_prefix]}")
print(f"  Model output entropy: {entropy:.3f} (higher = more uncertain)")
print(f"  Highest probability word: '{id2token[np.argmax(s_probs_weird)]}' ({np.max(s_probs_weird):.3f})")
print()
print("-> The student has never seen this kind of prefix during training! Output is very uncertain.")
print("-> Exposure Bias: training and inference prefix distributions are inconsistent.")


## 3. How OPD Works

OPD's core is just three steps:

```
Step 1: Student generates a response (rollout)
  Prompt: "I yesterday" -> Student generates: "I went to the park and felt happy"

Step 2: Teacher gives feedback at each of the Student's prefixes
  Position 2: prefix=[I, yesterday], Student wrote "went" -> Teacher: "OK"
  Position 3: prefix=[I, yesterday, went], Student wrote "park" -> Teacher: "Fine"
  Position 4: prefix=[I, yesterday, went, park], Student wrote "very" -> Teacher: "Not quite right"

Step 3: Update Student based on Teacher's feedback
  Words Teacher approved -> increase probability
  Words Teacher disapproved -> decrease probability
```

The key: the Student is corrected on **its own actually generated trajectories**.
Even if it produces a nonsensical prefix like "I yesterday very park", the Teacher still tells it what to do next at that nonsensical prefix.
This way, when the Student encounters the same nonsensical prefix during inference, it knows what to do.


In [ ]:
import numpy as np

# Demonstrate OPD core: Student does its own rollout, Teacher scores at Student's prefixes
print("=== OPD Core Flow Demonstration ===")
print()

prompt = [0, 1]  # "I yesterday"
sequence = prompt.copy()
temperature = 1.0

print(f"Prompt: {[id2token[i] for i in prompt]}")
print()

# Step 1: Student rollout (student writes on its own)
print("Step 1: Student autoregressive generation")
for step in range(3):
    logits = student_model(sequence)
    probs = softmax(logits / temperature)
    next_token = np.random.choice(vocab_size, p=probs)
    sequence.append(next_token)
    print(f"  Prefix={[id2token[i] for i in sequence[:-1]]} -> sampled {id2token[next_token]}")

print(f"\nGenerated sequence: {[id2token[i] for i in sequence]}")
print()

# Step 2: Teacher scores at each position
print("Step 2: Teacher evaluates at Student's own prefixes")
for t in range(len(prompt), len(sequence)):
    prefix = sequence[:t]
    sampled_token = sequence[t]
    
    s_logits = student_model(prefix)
    t_logits = teacher_model(prefix)
    s_logp = log_softmax(s_logits)[sampled_token]
    t_logp = log_softmax(t_logits)[sampled_token]
    
    advantage = t_logp - s_logp
    
    print(f"  Position {t}: prefix={[id2token[i] for i in prefix]}")
    print(f"    Student wrote '{id2token[sampled_token]}', student_logp={s_logp:.3f}, teacher_logp={t_logp:.3f}")
    print(f"    advantage = {advantage:+.3f} -> {'✅ bonus' if advantage > 0 else '❌ penalty'}")

print()
print("Key: No matter what nonsensical content the Student writes, the Teacher gives feedback on the actual prefix!")


## 4. Forward KL and Reverse KL

The difference between offline distillation and OPD can be understood through the **direction** of KL divergence.

KL divergence measures the "distance" between two distributions, but it is **asymmetric**:

$$D_{KL}(P \\| Q) \\neq D_{KL}(Q \\| P)$$

Just as "from home to school" and "from school to home" cover the same distance, but the uphill/downhill experience feels completely different.

**Forward KL (offline distillation)**:

$$D_{KL}(P_{Teacher} \\| P_{Student})$$

- Expectation taken under Teacher's distribution: where Teacher thinks is important, Student must learn well
- Behavior: **Mode-Covering** -- Student must cover all of Teacher's "modes"
- Problem: when small model capacity is limited, forcing coverage of all of teacher's modes may cause each mode to be learned poorly, and the output becomes more averaged

**Reverse KL (OPD)**:

$$D_{KL}(P_{Student} \\| P_{Teacher})$$

- Expectation taken under Student's distribution: only where Student frequently visits needs alignment with Teacher
- Behavior: **Mode-Seeking** -- Student only needs to find Teacher-approved modes that it can stably generate
- Intuition: reverse KL is more mode-seeking; the small model can prioritize aligning modes that it walks often and that teacher also approves; but this may also sacrifice diversity.


In [ ]:
import numpy as np

# Demonstrate Forward KL vs Reverse KL with real probability distributions
print('"=== Forward KL vs Reverse KL Intuition ==="')
print()

# Simulated Teacher's probability distribution over three styles
teacher_probs = np.array([0.40, 0.35, 0.25])
styles = ["Academic rigorous", "Humorous", "Concise direct"]

print("Teacher considers good answers to have three styles:")
for i, (style, prob) in enumerate(zip(styles, teacher_probs)):
    bar = "█" * int(prob * 40)
    print(f"  {style}: {prob:.0%} {bar}")
print()

# Forward KL (offline distillation): Student is required to cover all modes
# Use an "averaged" Student distribution to illustrate
student_avg_probs = np.array([0.33, 0.33, 0.34])  # Uniform-ish
forward_kl = np.sum(teacher_probs *
                    (np.log(teacher_probs + 1e-10)
                     - np.log(student_avg_probs + 1e-10)))

print("Forward KL (offline distillation) -- Student tries to cover all modes:")
for i, (style, prob) in enumerate(zip(styles, student_avg_probs)):
    bar = "█" * int(prob * 40)
    print(f"  {style}: {prob:.0%} {bar}")
print(f"  Forward KL = {forward_kl:.4f}")
print(f"  -> Output becomes a mishmash: 'According to relevant research, haha, simply put...'")
print()

# Reverse KL (OPD): Student specializes in a mode it's good at
student_spec_probs = np.array([0.05, 0.85, 0.10])  # Specialize in humorous
reverse_kl = np.sum(student_spec_probs *
                     (np.log(student_spec_probs + 1e-10)
                      - np.log(teacher_probs + 1e-10)))

print("Reverse KL (OPD) -- Student specializes in its strongest mode:")
for i, (style, prob) in enumerate(zip(styles, student_spec_probs)):
    bar = "█" * int(prob * 40)
    print(f"  {style}: {prob:.0%} {bar}")
print(f"  Reverse KL = {reverse_kl:.4f}")
print(f"  -> Stable, high-quality humorous responses")
print()

print(f"Comparison: Forward KL ({forward_kl:.4f}) vs Reverse KL ({reverse_kl:.4f})")
print("OPD is friendlier to small models: no need to cover everything, specializing in one area is enough!")


## 5. OPSD Without an External Teacher

OPD requires a large Teacher model, but OPSD (On-Policy Self-Distillation) does not -- **Teacher and Student are the same model, except the Teacher has access to additional information.**

```
OPSD's core setup:

Student (closed-book): Only sees the question
  -> Writes its own answer

Teacher (open-book): Same model, but has seen the gold answer/solution steps
  -> At the Student's prefixes, gives "if I knew the answer, how I would write the next step"

Training goal: Bring the closed-book Student closer to the open-book Teacher
-> Internalize the 'only know with answer' ability into 'can do without answer'
```

**Intuition**: When doing math problems, looking at the answer you think "oh, so that's how", but close the answer and you forget.
OPSD repeatedly practices having the "looking-at-answer self" teach the "closed-book self".

**Limitation**: If the additional information the Teacher sees can never be obtained at test time (e.g., problem-specific standard solutions),
OPSD may only learn an "averaged problem-solving strategy" rather than genuine reasoning ability.
So OPSD is better suited for internalizing **shared rules** (format preferences, system prompts) rather than **problem-specific information**.


## 6. Three Levels of Supervision Granularity

When the Teacher gives feedback, it can provide information at different "granularity" levels:

| Granularity | What the teacher tells you | Information amount | Computation cost |
|:---|:---|:---|:---|
| **sampled-token** | Only tells you "is the word you chose good or not" | Least | Smallest |
| **top-k** | Tells you "the K best words in my opinion" | Medium | Medium |
| **full-vocab** | Tells you "the probability of every word in the vocabulary" | Most | Largest (32000 dims!) |

```
sampled-token:  Student wrote "park" -> Teacher: "OK, 6 points"
top-k (k=5):    Student wrote "park" -> Teacher: "Top 5 are: school(9pts) park(6pts) store(4pts) home(3pts) mall(2pts)"
full-vocab:     Student wrote "park" -> Teacher: "32000 words each scored..."
```


In [ ]:
import numpy as np

# Compare three levels of Teacher feedback granularity
prefix_demo = [0, 1, 2]  # "I went yesterday"

print("=== Three Signal Granularity Levels Compared ===")
print()

# Sampled-token: only look at the chosen word
t_logits = teacher_model(prefix_demo)
t_logprobs = log_softmax(t_logits)
sampled_token = 3  # "school"

print(f"1. sampled-token:")
print(f"   Teacher only tells you the score for '{id2token[sampled_token]}': {t_logprobs[sampled_token]:.3f}")
print(f"   You don't know the situation for the other 9 words")
print()

# Top-k: see Teacher's top K preferred words
k = 5
topk_idx = np.argsort(t_logprobs)[-k:][::-1]
print(f"2. top-{k}:")
for idx in topk_idx:
    print(f"   Teacher considers '{id2token[idx]}' score: {t_logprobs[idx]:.3f}")
print()

# Full-vocab
print(f"3. full-vocab:")
print(f"   Teacher tells you the probability distribution of all {vocab_size} words")
all_probs = softmax(t_logits)
for i, tok in id2token.items():
    bar = "█" * int(all_probs[i] * 40)
    print(f"   {tok:4s}: {all_probs[i]:.4f}  {bar}")


## 7. Sampled-Token KL Estimators

Sampled-token saves the most computation, but we only have information about one token, yet need to estimate the KL divergence of the entire distribution. What to do?

The true Reverse KL requires the full distribution:

$$KL(P_S \\| P_T) = \\sum_i P_S(i) \\cdot \\log\\frac{P_S(i)}{P_T(i)}$$

But now we only have one sampled token `y`. Three estimation methods:

| Estimator | Formula | Characteristics |
|:---|:---|:---|
| **k1** | $\\log P_S(y) - \\log P_T(y)$ | Unbiased, but high variance, can be negative |
| **k2** | $\\frac{1}{2}(\\log P_S(y) - \\log P_T(y))^2$ | Always positive, biased but stable |
| **k3** | $\\frac{P_T(y)}{P_S(y)} - \\log\\frac{P_T(y)}{P_S(y)} - 1$ | Non-negative, commonly used as a low-variance alternative estimator; whether it applies depends on the sampling distribution and target |


In [ ]:
import numpy as np

def k1_estimator(s_logp, t_logp):
    """Direct difference: unbiased but high variance, can be negative"""
    return s_logp - t_logp

def k2_estimator(s_logp, t_logp):
    """Squared approximation: biased but always positive, low variance"""
    diff = s_logp - t_logp
    return 0.5 * diff * diff

def k3_estimator(s_logp, t_logp):
    """k3: non-negative, commonly used as a low-variance alternative estimator (whether unbiased depends on the target and sampling distribution)"""
    ratio = t_logp - s_logp  # log(P_T / P_S)
    r = np.exp(ratio)         # P_T / P_S
    return r - ratio - 1

# Compare three estimators
print("=== Three KL Estimators Compared ===")
print(f"{'student logp':>12} {'teacher logp':>12} {'k1':>10} {'k2':>10} {'k3':>10}  Note")
print("-" * 65)

cases = [
    (-0.5, -0.5, "Identical"),
    (-1.0, -0.5, "Teacher prefers"),
    (-0.5, -1.0, "Student prefers"),
    (-2.0, -0.5, "Large gap - Teacher"),
    (-0.5, -2.0, "Large gap - Student"),
    (-3.0, -0.1, "Extreme gap"),
]

for s_lp, t_lp, desc in cases:
    k1 = k1_estimator(s_lp, t_lp)
    k2 = k2_estimator(s_lp, t_lp)
    k3 = k3_estimator(s_lp, t_lp)
    print(f"{s_lp:>+10.2f} {t_lp:>+10.2f} {k1:>+10.4f} {k2:>10.4f} {k3:>10.4f}  ← {desc}")

print()
print("k1 can be negative, so it cannot be used directly as a loss")
print("k2 always positive -> biased but stable")
print("k3 non-negative, commonly used as a more stable KL-related estimator; confirm sampling distribution and optimization target in practice")


## 8. A Complete Training Loop

We've covered the individual components of OPD: Student self-generation, Teacher scoring, Teacher grading, Student updating. Now let's chain them into one complete training iteration.

One iteration's steps: take a prompt -> Student generates its own response -> Teacher generates a reference response for the same prompt -> Teacher compares the two responses and gives sentence-by-sentence annotations -> organize annotations into SFT training data -> update Student. In the next iteration, the Student regenerates from updated parameters, and the Teacher grades again. The key to this loop is: the Student is corrected at the new state after making mistakes, rather than memorizing someone else's gold answers.

Below, the entire loop is written as a runnable training loop.


In [ ]:
import numpy as np

def simulate_opd_training(prompt_ids, num_gen_tokens=3, mode="sampled_token_k3", topk=5):
    """Simulate one complete OPD training step"""
    
    # Step 1: Student rollout (student generates on its own)
    sequence = list(prompt_ids).copy()
    temp = 1.0
    
    print("Step 1: Rollout (student autoregressive generation)")
    print(f"  Prompt: {[id2token[i] for i in prompt_ids]}")
    for step in range(num_gen_tokens):
        logits = student_model(sequence)
        probs = softmax(logits / temp)
        next_token = np.random.choice(vocab_size, p=probs)
        sequence.append(next_token)
    
    generated = sequence[len(prompt_ids):]
    print(f"  Generated: {[id2token[i] for i in generated]}")
    
    # Step 2: Compute OPD loss at each position
    print(f"\nStep 2: Compute OPD loss (mode: {mode})")
    total_loss = 0
    
    for t in range(len(prompt_ids), len(sequence)):
        prefix = sequence[:t]
        sampled_token = sequence[t]
        
        if mode == "sampled_token_k3":
            s_logp = log_softmax(student_model(prefix))[sampled_token]
            t_logp = log_softmax(teacher_model(prefix))[sampled_token]
            loss = k3_estimator(s_logp, t_logp)
            print(f"  position {t}: '{id2token[sampled_token]}'"
                  f" | s_logp={s_logp:.3f} t_logp={t_logp:.3f}"
                  f" | k3={loss:.4f}")
        
        elif mode == "topk_rkl":
            t_logp = log_softmax(teacher_model(prefix))
            topk_idx = np.argsort(t_logp)[-topk:][::-1]
            t_renorm = softmax(t_logp[topk_idx])
            s_logp = log_softmax(student_model(prefix))
            s_renorm = softmax(s_logp[topk_idx])
            loss = np.sum(s_renorm * (np.log(s_renorm + 1e-10) - np.log(t_renorm + 1e-10)))
            print(f"  Position {t}: top-{topk}={[id2token[i] for i in topk_idx]} | RKL={loss:.4f}")
        
        total_loss += loss
    
    avg_loss = total_loss / len(generated)
    print(f"\nTotal loss: {total_loss:.4f} | Average: {avg_loss:.4f}")
    print(f"-> Backpropagate gradients, update Student -> next rollout -> loop")
    return avg_loss

np.random.seed(42)
print("### sampled_token + k3 mode ###")
simulate_opd_training([0, 1], num_gen_tokens=3, mode="sampled_token_k3")


In [ ]:
import numpy as np

np.random.seed(42)
print("### top-k Reverse KL mode ###")
simulate_opd_training([0, 1], num_gen_tokens=3, mode="topk_rkl", topk=5)

## 9. Conditions for Development and Production Use

The idea behind OPD is not new, but repeatedly combining student rollout, teacher scoring, and gradient updates used to be difficult to engineer. Several infrastructure advances have made it more practical:

1. **Unified training and inference frameworks** such as verl, vLLM, and DeepSpeed connect rollout, scoring, and optimization in one pipeline.
2. **Cross-tokenizer distillation** reduces dependence on identical vocabularies through alignment, top-k probabilities, sampled tokens, or text feedback, though approximation errors remain.
3. **MoE teachers** can improve teacher capacity per unit of active compute and support large-scale sampling and scoring.

MoE is engineering support, not the source of OPD's mathematical advantage. It can make a teacher stronger or cheaper, but does not explain why on-policy distillation differs from offline distillation.

A careful statement is that **on-policy teacher–student methods are entering post-training experiments and engineering discussions, although organizations do not publish them under one consistent name**. Reports may say distillation, expert distillation, self-distillation, or simply “student rollout plus teacher feedback.”

| System | Publicly confirmed practice | How to interpret it |
|:---|:---|:---|
| **Gemma 2** | Its paper explicitly discusses knowledge distillation | KD is an important tool for training smaller models |
| **Qwen3** | Its report describes multi-stage post-training and distillation-related thinking/non-thinking flows | Distillation compresses reasoning and general ability into one model |

Do not memorize a fixed pipeline such as `SFT -> RL -> OPD`. A more accurate engineering map is:

```text
cold start / SFT
  -> RL or verifiable-task training
  -> sampling, filtering, rejection sampling
  -> distillation / self-distillation / OPD into the target model
```

Teams change the order according to data, compute, model size, and deployment goals. OPD's value is that the student visits its own distribution and receives dense feedback at states it will actually encounter.

References: [Gemma 2 paper](https://arxiv.org/abs/2408.00118) and [Qwen3 technical report](https://arxiv.org/abs/2505.09388). Details unsupported by a paper, report, or model card should be treated as leads rather than official training pipelines.


## 10. Paper Map

The following papers have public sources and directly support this notebook's main thread:

| Paper | Main contribution | Source |
|:---|:---|:---|
| **MiniLLM** (2023) | Reverse-KL distillation for generative LMs; an early representative of OPD ideas | [arXiv](https://arxiv.org/abs/2306.08543) |
| **GKD** (2023/2024) | Unifies on-policy and off-policy generative distillation and emphasizes learning on student generations | [arXiv](https://arxiv.org/abs/2306.13649) |
| **Self-Distilled Reasoner / OPSD** (2026) | Uses one model with privileged information as teacher and without it as student | [arXiv](https://arxiv.org/abs/2605.18141) |
| **Pitfalls of On-Policy Self-Distillation** (2026) | Studies unreliable teachers, unstable gradients, and hard-to-internalize privileged information | [arXiv](https://arxiv.org/abs/2605.11182) |
| **EDGE-OPD** (2026) | Uses early-draft guidance to improve on-policy distillation efficiency | [arXiv](https://arxiv.org/abs/2605.23493) |

OPD and OPSD are not universal solutions. Their advantage is **on-policy prefixes plus dense token-level feedback**; their risks include unreliable teacher distributions and privileged information that cannot be reconstructed at test time.

When reading an OPD paper, first ask: **who is the teacher, and what is the objective?**

### 10.1 By Teacher Type

| Teacher type | Available signal | Typical route |
|:---|:---|:---|
| **External white-box** | logits or top-k probabilities | White-box KL distillation such as MiniLLM and GKD |
| **External black-box** | sampled answers only | Samples, scorers, or reward models approximate teacher feedback |
| **Self-teacher** | the same model under privileged context | OPSD / self-distillation |
| **Multiple teachers** | expert models or checkpoints | Ability merging, compression, and continual learning |

If teacher logits are available, white-box KL gives the densest signal. Otherwise, use black-box distillation, a reward model, or self-distillation.

### 10.2 By Primary Objective

| Objective | Typical use |
|:---|:---|
| **Compression / strong-to-weak transfer** | A large model teaches a smaller model |
| **Post-RL consolidation** | Compress RL-improved behavior into a stable target model |
| **Self-distillation** | An open-book version teaches the closed-book version without an external teacher |
| **Continual learning** | Consolidate online interactions or new-task experience into weights |
| **Efficiency** | Reduce teacher calls or rollout cost |

One method may belong to several categories; OPSD, for example, is self-distillation and may also compress reasoning ability.


## Summary

### Distribution Perspective: Connections and Differences Among SFT, RL, and OPD (Opening)

1. ✅ **Model distribution**: A language model is essentially a probability distribution generator -- post-training is about reshaping this distribution
2. ✅ **SFT** = Imitating external data distributions: target distribution from external datasets, strong for cold starting and format shaping, weak for over-pulling
3. ✅ **RL** = Selecting high-reward directions from own behavior: samples from model's own generation (on-policy), better for verifiable tasks, but reward is sparse
4. ✅ **OPD** = Student walks on its own, Teacher corrects where the student arrives: combines on-policy relevance with distillation's dense signal
5. ✅ **Core differences**: Data source (external vs. self-generated), signal density (dense vs. sparse), distribution shifting method (overall pull vs. local filtering/correction)

### Core Principles (Sections 1-9)

6. ✅ **Knowledge distillation** = Large model teaches small model
7. ✅ **Offline distillation** = Memorizing teacher's model essays -> prefix from teacher/dataset
8. ✅ **Exposure Bias** = Training prefix and inference prefix are inconsistent -> collapse when deviating during inference
9. ✅ **OPD** = Student does its own rollout, Teacher gives feedback at Student's prefixes
10. ✅ **Forward KL** (offline distillation) = Mode-Covering -> small model learns everything, masters nothing
11. ✅ **Reverse KL** (OPD) = Mode-Seeking -> small model specializes in one area
12. ✅ **OPSD** = No external Teacher needed, same model's open-book version teaches closed-book version
13. ✅ **Three granularity levels**: sampled-token < top-k < full-vocab
14. ✅ **Three KL estimators**: k1 (unbiased, can be negative), k2 (biased, always positive), k3 (unbiased, always positive, recommended)

### Engineering and Ecosystem (Sections 10-15)

15. ✅ **Engineering infrastructure**: verl + vLLM + DeepSpeed + cross-tokenizer alignment enable OPD deployment
16. ✅ **Paper quick look**: GKD / OPSD / Pitfalls / OGLS-SD as core papers
17. ✅ **Paper landscape**: Categorize by foundation methods / self-distillation / efficiency stability / industrial practice, more useful than memorizing paper names
18. ✅ **Classification dimensions**: By Teacher type (white-box/black-box/self-teacher/context/multi-teacher) and by objective (compression/integration/continual learning/RL replacement)
19. ✅ **Industrial practice**: Gemma 2 / Qwen3 and other public materials show distillation-related approaches; OPD-related systems depend on whether there are official sources, but don't write all systems as the same fixed pipeline
20. ✅ **Model formats**: Safetensors (safe loading), GGUF (quantized inference), ONNX (cross-platform)
21. ✅ **HuggingFace**: Center of open-source models, understanding naming conventions is the first step to finding models
22. ✅ **Deployment tools**: Without GPU, first look at llama.cpp/ollama; with GPU and high throughput, commonly look at vLLM; NVIDIA maximum optimization, look at TensorRT-LLM; finally choose based on hardware and business testing

### One-Sentence Summary

SFT imitates external data distributions, RL selects using reward within its own behavior, and OPD has the student walk on its own while the teacher corrects where the student arrives.

The three are not replacements, but three different distribution shifting methods. Offline distillation = memorizing teacher's standard answers; OPD = student writes on its own, teacher grades sentence by sentence. Offline distillation may encounter training/inference prefix inconsistency; OPD's advantage is making teacher feedback closer to the positions the student actually visits.

OPD is not a silver bullet -- the real advantage is **on-policy prefix + dense token-level feedback**, the real risk is **unreliable teacher distribution + PI cannot be internalized**.
Which path to choose depends on your constraints: have logits -> white-box, only have API -> black-box, no external teacher -> self-distillation.

### References

- awesome-on-policy-distillation: https://github.com/chrisliu298/awesome-on-policy-distillation
- OPD Survey: https://arxiv.org/abs/2604.00626
- MiniLLM: https://arxiv.org/abs/2306.08543
- GKD: https://arxiv.org/abs/2306.13649
- ExOPD: https://arxiv.org/abs/2602.12125
- OPSD: https://arxiv.org/abs/2601.18734
- Pitfalls (Zhu et al.): https://arxiv.org/abs/2605.11182
- OGLS-SD: https://arxiv.org/abs/2605.12400
- Thinking Machines blog: https://thinkingmachines.ai/blog/on-policy-distillation/
- TRL DistillationTrainer: https://huggingface.co/docs/trl


## Exercises

> You can ask an AI to explain the ideas or check your direction, but don't have it "solve the exercise" for you.

**Exercise 1: Distribution shifting directions of three training methods** Understanding SFT, RL, and OPD from a distribution perspective:- **SFT**: the student imitates the external data distribution $p_{data}$- **RL**: the student selects high-reward directions within its own generation distribution- **OPD**: the student accepts Teacher corrections on its own trajectories Analyze whether the following statement is correct:> "SFT's training data comes from an external dataset, so the input distribution during training (the source of prompts) may be inconsistent with the distribution during inference. OPD lets the student learn on its own generated sequences, avoiding this distribution inconsistency." Hint: this is the core of Exposure Bias -- during training the student sees "perfect answers", during inference the student sees "its own generated (possibly erroneous) preceding context".


In [ ]:
# Exercise 1: direction of distribution movement under three training methods
# Decide whether the statement above is correct

answer = None  # enter True or False here

assert answer is not None, "Enter True or False"
assert isinstance(answer, bool), "Enter True or False"

if answer is True:
    print("Exercise 1 passed:")
    print("   SFT has Exposure Bias: its training data contains perfect prefixes written by humans,")
    print("   while at inference the model sees prefixes it generated itself, which may contain errors; the distributions differ.")
    print("   OPD corrects the student along its own trajectories, aligning the training and inference input distributions")
    print("   and addressing Exposure Bias at its source.")
else:
    print("The statement is correct.")
    print("Hint: compare where the model's prefixes come from during SFT training and inference.")


**Exercise 2: Forward KL vs Reverse KL** KL divergence has two directions, corresponding to different optimization behaviors:- **Forward KL**: $D_{KL}(P \\| Q) = \\sum P_i \\log(P_i / Q_i)$, minimizing it makes Q "cover" all modes of P (mean-seeking)- **Reverse KL**: $D_{KL}(Q \\| P) = \\sum Q_i \\log(Q_i / P_i)$, minimizing it makes Q "lock onto" a certain mode of P (mode-seeking) Suppose P is a bimodal distribution (two Gaussian peaks), Q is a unimodal distribution. Analyze by drawing where the peak of Q will appear under Forward KL and Reverse KL respectively. Hint: Forward KL -> Q's peak will appear between the two peaks of P (covering both peaks); Reverse KL -> Q's peak will align with one peak of P (locking onto one peak).


In [ ]:
# Demonstrate Forward KL vs Reverse KL with real probability distributions
# A) Forward KL makes the Student more concentrated (mode-seeking)
# B) Reverse KL makes the Student cover every Teacher mode (mean-seeking)
# C) Forward KL is better for SFT because the Student should learn the Teacher broadly
# D) Both KL directions produce exactly the same optimum

answer = "_"  # enter A / B / C / D here

assert answer in "ABC", "Please fill in one of A/B/C"

if answer == "C":
    print("Exercise 2 passed:")
    print("   Forward KL is mean-seeking, so the Student tries to cover all of the Teacher's output modes,")
    print("   including modes with low probability.")
    print("   SFT uses Forward KL so the Student learns the Teacher's capabilities broadly.")
    print("   Reverse KL is mode-seeking, concentrating the Student in the Teacher's highest-probability region.")
    print("   This suits RL optimization toward high reward, but may omit long-tail capabilities.")
else:
    print(f"You chose {answer}.")
    print("Hint: Forward KL covers broadly for learning; Reverse KL focuses on the optimum for optimization.")


**Exercise 3: KL Estimators Compared** OPD needs to estimate KL divergence to control how far the student distribution deviates from the reference distribution. Three commonly used KL estimators:- $k_1$: unbiased estimate, but can be negative- $k_2$: biased, but always positive- $k_3$: non-negative, more stable than $k_2$ Analyze: why is it a problem that a KL estimator can be negative? What consequences does a negative "KL value" lead to during training? Hint: if the estimated KL value is negative, the model thinks it is "very close" to the reference distribution, so it relaxes the constraint, when in fact it may have already deviated far.


In [ ]:
# Exercise 3: compare KL estimators
# A) KL divergence is non-negative; a negative estimate is merely harmless estimation error
# B) A negative KL estimate fools the KL penalty into believing the model has not moved from the reference,
#    relaxing the constraint and allowing further drift
# C) Only k1 can be negative, but it has the lowest variance, so k1 alone is sufficient in practice
# D) Estimator bias and variance do not affect training; only learning rate matters

answer = "_"  # enter A / B / C / D here

assert answer in "ABC", "Please fill in one of A/B/C"

if answer == "B":
    print("Exercise 3 passed:")
    print("   The KL penalty prevents the student distribution from moving too far from the reference.")
    print("   If its estimate is negative, the penalty fails and the model believes it is still close,")
    print("   even when it may already have drifted badly through reward hacking.")
    print("   k2 and k3 avoid the issue by guaranteeing nonnegative weights,")
    print("   This introduces some bias, but training stability matters more.")
else:
    print(f"You chose {answer}.")
    print("Hint: the KL penalty acts as a safety rope during training.")
